# Basic localization-signal diagnostic

This notebook asks one focused question: **can position be decoded from AE latents and GNN embeddings, and what changes after update adaptation?** It includes the saved bootstrap/reference-only baseline as well as online round 1 and post-transition round 2.

For each saved run, it compares the bootstrap representation with online round 1 and post-transition round 2. The bootstrap stage has a held-out reference comparison only because update scans have not yet entered the pipeline. The same reference location split is used at every stage.

The update-coordinate labels are evaluation truth in the saved pipeline. Here, they are used only for this diagnostic: a subset fits a temporary localization MLP, and the disjoint held-out subset scores it. No AE, GNN, updater, or saved pipeline artifact is retrained or changed.

The bootstrap GNN embeddings are a **transductive baseline**: the saved GNN was trained on all reference nodes, whose input features include their coordinates. The held-out probe split does not hide those coordinates from the GNN. The same limitation applies to held-out-reference rows in later stages.


In [1]:
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display

from ae import encode_prepared_dataset, load_encoder_features
from gufu_data import load_prepared_dataset
from gufu_debug_evaluation import _latest_snapshot
from gufu_evaluation import train_reference_location_mlp_predictions
from spatial_normalization import load_spatial_transform
from lp_streaming import load_streaming_lp_input
from streaming_gnn import load_streaming_gnn_state
from streaming_update import load_streaming_online_graph

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "gufu.py").is_file():
    raise RuntimeError("Run this notebook with the GUFU repository root as the working directory.")

OUTPUT_ROOTS = {
    "GF_from1F_12": PROJECT_ROOT / "output_from1F_12",
    "GF_fromGF_12": PROJECT_ROOT / "output_fromGF_12",
}

TRAIN_FRACTION = 0.75
SPLIT_SEED = 17
PROBE_EPOCHS_OVERRIDE = None  # None uses the saved updater probe epoch count.
DEVICE = "cpu"

STAGES = ("online_round1", "post_transition_round2")


## What is trained and what is scored

For each representation, a fresh MLP uses the saved run's hidden width and learning rate. The bootstrap row uses the original saved encoder and GNN feature artifacts and reports held-out-reference performance only. The online and post-transition rows also report reference-to-update transfer and update-to-update diagnostic performance.

1. **Held-out reference:** fit on 75% of reference coordinate groups; score on the other 25%.
2. **Reference → update:** use that same reference-trained probe; score on the held-out update groups.
3. **Update → update:** fit a separate temporary probe on 75% of update coordinate groups; score on the same held-out update groups as protocol 2.

Scans with exactly the same coordinates stay in one split, so repeated scans at a location cannot cross from probe training to scoring. This is a random location-group split, not a spatial-block split; nearby but distinct locations can still occur on both sides.

“AE latent” means the encoder output from the AE checkpoint used at that stage. “GNN embedding” means the saved GNN output for that stage. The update AE vectors are loaded from the saved encoder artifact; reference AE vectors are computed in inference mode into a temporary directory. The saved pipeline artifacts remain frozen.


In [2]:
def _as_numpy(value):
    if hasattr(value, "detach"):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def _split_coordinate_groups(xy, *, fraction=TRAIN_FRACTION, seed=SPLIT_SEED):
    xy = np.asarray(xy, dtype=np.float64)
    unique_xy, group_ids = np.unique(xy, axis=0, return_inverse=True)
    if len(unique_xy) < 2:
        raise ValueError("Need at least two distinct coordinate groups for a held-out split.")
    rng = np.random.default_rng(seed)
    group_order = rng.permutation(len(unique_xy))
    n_train_groups = min(
        max(1, int(np.floor(fraction * len(unique_xy)))),
        len(unique_xy) - 1,
    )
    train_groups = group_order[:n_train_groups]
    train_mask = np.isin(group_ids, train_groups)
    train_idx = np.flatnonzero(train_mask)
    test_idx = np.flatnonzero(~train_mask)
    return train_idx, test_idx, n_train_groups, len(unique_xy) - n_train_groups


def _load_bootstrap_reference_data(run_name, output_root):
    # These are the original reference-only outputs created before streaming updates.
    _snapshot_dir, snapshot_manifest, run_root = _latest_snapshot(output_root)
    prepared = load_prepared_dataset(run_root / "prepared_bootstrap.json")
    encoder = load_encoder_features(run_root / "encoder_features.json")
    lp_manifest, lp_arrays = load_streaming_lp_input(
        run_root / "link_prediction_input"
    )

    from gufu_artifacts import sha256_file
    ae_checkpoint = run_root / "checkpoints" / "ae.pt"
    gnn_checkpoint = run_root / "checkpoints" / "gnn.pt"
    if encoder.checkpoint_sha256 != sha256_file(ae_checkpoint):
        raise ValueError(f"{run_name}: bootstrap AE features do not match the saved AE checkpoint.")
    if lp_manifest.gnn_checkpoint_sha256 != sha256_file(gnn_checkpoint):
        raise ValueError(f"{run_name}: bootstrap GNN features do not match the saved GNN checkpoint.")
    if lp_manifest.encoder_checkpoint_sha256 != encoder.checkpoint_sha256:
        raise ValueError(f"{run_name}: bootstrap GNN and AE artifacts use different AE checkpoints.")
    if lp_manifest.batch_role != "bootstrap_reference":
        raise ValueError(f"{run_name}: saved bootstrap GNN input is not reference-only.")

    reference_ids = tuple(item.scan_id for item in prepared.reference_positions)
    reference_xy = np.asarray(
        [(item.x, item.y) for item in prepared.reference_positions],
        dtype=np.float64,
    )
    ae_by_id = {
        scan_id: row
        for scan_id, row in zip(encoder.scan_ids, encoder.feature_matrix)
    }
    gnn_rows = {node_id: row for row, node_id in enumerate(lp_manifest.node_ids)}
    if any(scan_id not in ae_by_id or scan_id not in gnn_rows for scan_id in reference_ids):
        raise ValueError(f"{run_name}: bootstrap reference IDs do not align across artifacts.")

    reference = {
        "ids": reference_ids,
        "xy": reference_xy,
        "ae": np.asarray([ae_by_id[scan_id] for scan_id in reference_ids], dtype=np.float32),
        # The compact LP bundle stores the bootstrap GNN node embeddings.
        "gnn": np.asarray(
            [lp_arrays.node_features[gnn_rows[scan_id]] for scan_id in reference_ids],
            dtype=np.float32,
        ),
    }
    probe_config = snapshot_manifest["pipeline_config"]["fingerprint_update"]
    probe = {
        "hidden_dim": int(probe_config["hidden_dim"]),
        "epochs": (
            int(PROBE_EPOCHS_OVERRIDE)
            if PROBE_EPOCHS_OVERRIDE is not None
            else int(probe_config["updater_epochs"])
        ),
        "learning_rate": float(probe_config["updater_learning_rate"]),
        "seed": int(snapshot_manifest["pipeline_config"]["seed"]) + 1701,
    }
    spatial_transform = load_spatial_transform(
        run_root / "artifacts" / "spatial" / "GF.json"
    )
    return reference, probe, spatial_transform
def _run_bootstrap_probe_comparisons(run_name, reference, probe, spatial_transform):
    train_idx, test_idx, train_groups, test_groups = _split_coordinate_groups(reference["xy"])
    rows = []
    for representation in ("ae", "gnn"):
        prediction = train_reference_location_mlp_predictions(
            reference[representation][train_idx],
            reference["xy"][train_idx],
            reference[representation][test_idx],
            spatial_transform=spatial_transform,
            hidden_dim=probe["hidden_dim"],
            epochs=probe["epochs"],
            learning_rate=probe["learning_rate"],
            seed=probe["seed"],
            device=DEVICE,
        )
        rmse, mean_error = _score(prediction, reference["xy"][test_idx])
        rows.append({
            "run": run_name,
            "stage": "bootstrap_reference_only",
            "representation": representation,
            "protocol": "held_out_reference",
            "train_scans": len(train_idx),
            "train_locations": train_groups,
            "test_scans": len(test_idx),
            "test_locations": test_groups,
            "position_rmse": rmse,
            "mean_position_error": mean_error,
        })
    return rows


def _load_stage_data(run_name, output_root, stage):
    snapshot_dir, snapshot_manifest, run_root = _latest_snapshot(output_root)
    if stage == "online_round1":
        graph_relative = "online_graph"
        gnn_relative = "online_gnn"
        encoder_relative = "update_encoder_features.json"
        reference_prepared_relative = "reference_prepared_before.json"
        reference_provenance_relative = "reference_value_provenance_before.json"
        ae_checkpoint = snapshot_dir / "source_ae_checkpoint.pt"
    elif stage == "post_transition_round2":
        graph_relative = "post_transition_fingerprint_update_round2/query_graph"
        gnn_relative = "post_transition_fingerprint_update_round2/query_gnn"
        encoder_relative = "post_transition_fingerprint_update_round2/update_encoder_features.json"
        reference_prepared_relative = snapshot_manifest["current_state"]["reference_prepared"]
        reference_provenance_relative = snapshot_manifest["current_state"]["reference_value_provenance"]
        ae_checkpoint = snapshot_dir / snapshot_manifest["current_state"]["ae_checkpoint"]
    else:
        raise ValueError(f"Unknown stage: {stage}")

    graph_dir = snapshot_dir / graph_relative
    graph = load_streaming_online_graph(graph_dir)
    gnn_state = load_streaming_gnn_state(
        snapshot_dir / gnn_relative, graph_dir, device=DEVICE
    )
    reference_prepared_path = snapshot_dir / reference_prepared_relative
    reference_provenance_path = snapshot_dir / reference_provenance_relative
    reference_prepared = load_prepared_dataset(
        reference_prepared_path,
        value_provenance_path=(
            reference_provenance_path if reference_provenance_path.is_file() else None
        ),
    )
    update_prepared = load_prepared_dataset(snapshot_dir / "input_prepared_batch.json")
    update_encoder = load_encoder_features(snapshot_dir / encoder_relative)

    # Verify the AE checkpoint and saved update encodings refer to the same feature space.
    from gufu_artifacts import sha256_file
    if update_encoder.checkpoint_sha256 != sha256_file(ae_checkpoint):
        raise ValueError(f"{run_name}/{stage}: update encodings do not match the selected AE checkpoint.")

    # Encode the reference scans with that exact AE checkpoint without writing into the run.
    with tempfile.TemporaryDirectory(prefix="gufu_basic_eval_") as temporary_directory:
        reference_encoder = encode_prepared_dataset(
            reference_prepared,
            ae_checkpoint,
            Path(temporary_directory) / "reference_encoder_features.json",
            batch_size=256,
            device=DEVICE,
        )
        reference_ae_by_id = {
            scan_id: row
            for scan_id, row in zip(reference_encoder.scan_ids, reference_encoder.feature_matrix)
        }

    update_ae_by_id = {
        scan_id: row
        for scan_id, row in zip(update_encoder.scan_ids, update_encoder.feature_matrix)
    }

    node_rows = {node_id: row for row, node_id in enumerate(graph.manifest.node_ids)}
    node_embeddings = _as_numpy(gnn_state.node_embeddings)
    reference_ids = tuple(graph.manifest.labeled_reference_ids)
    reference_xy = np.asarray(
        graph.manifest.labeled_reference_coordinates, dtype=np.float64
    )
    if len(reference_ids) != len(reference_xy):
        raise ValueError(f"{run_name}/{stage}: reference IDs and coordinates do not align.")
    if any(scan_id not in node_rows or scan_id not in reference_ae_by_id for scan_id in reference_ids):
        raise ValueError(f"{run_name}/{stage}: missing a labeled reference representation.")

    update_ids = tuple(graph.manifest.update_sample_ids)
    if set(update_ids) != set(update_ae_by_id):
        raise ValueError(f"{run_name}/{stage}: update AE rows and graph update IDs differ.")
    update_truth_by_id = {
        item.scan_id: (item.x, item.y)
        for item in update_prepared.evaluation_positions
    }
    paired_update_ids = tuple(
        scan_id for scan_id in update_ids if scan_id in update_truth_by_id
    )
    if not paired_update_ids:
        raise ValueError(f"{run_name}/{stage}: no update scans have evaluation coordinates.")
    if any(scan_id not in node_rows for scan_id in paired_update_ids):
        raise ValueError(f"{run_name}/{stage}: an update scan is missing from the GNN graph.")

    reference = {
        "ids": reference_ids,
        "xy": reference_xy,
        "ae": np.asarray([reference_ae_by_id[scan_id] for scan_id in reference_ids], dtype=np.float32),
        "gnn": np.asarray([node_embeddings[node_rows[scan_id]] for scan_id in reference_ids], dtype=np.float32),
    }
    update = {
        "ids": paired_update_ids,
        "xy": np.asarray([update_truth_by_id[scan_id] for scan_id in paired_update_ids], dtype=np.float64),
        "ae": np.asarray([update_ae_by_id[scan_id] for scan_id in paired_update_ids], dtype=np.float32),
        "gnn": np.asarray([node_embeddings[node_rows[scan_id]] for scan_id in paired_update_ids], dtype=np.float32),
    }

    spatial_transform = load_spatial_transform(run_root / "artifacts" / "spatial" / "GF.json")
    probe_config = snapshot_manifest["pipeline_config"]["fingerprint_update"]
    probe_epochs = (
        int(PROBE_EPOCHS_OVERRIDE)
        if PROBE_EPOCHS_OVERRIDE is not None
        else int(probe_config["updater_epochs"])
    )
    return reference, update, spatial_transform, {
        "hidden_dim": int(probe_config["hidden_dim"]),
        "epochs": probe_epochs,
        "learning_rate": float(probe_config["updater_learning_rate"]),
        "seed": int(snapshot_manifest["pipeline_config"]["seed"]) + 1701,
    }


def _score(predicted_xy, truth_xy):
    radial_error = np.linalg.norm(
        np.asarray(predicted_xy, dtype=np.float64) - np.asarray(truth_xy, dtype=np.float64),
        axis=1,
    )
    return float(np.sqrt(np.mean(radial_error**2))), float(np.mean(radial_error))


def _run_probe_comparisons(run_name, stage, reference, update, spatial_transform, probe):
    reference_train, reference_test, reference_train_groups, reference_test_groups = (
        _split_coordinate_groups(reference["xy"])
    )
    update_train, update_test, update_train_groups, update_test_groups = (
        _split_coordinate_groups(update["xy"])
    )
    rows = []

    for representation in ("ae", "gnn"):
        reference_features = reference[representation]
        update_features = update[representation]

        # One reference-trained probe scores both held-out references and held-out updates.
        reference_queries = np.concatenate(
            (reference_features[reference_test], update_features[update_test]), axis=0
        )
        reference_predictions = train_reference_location_mlp_predictions(
            reference_features[reference_train],
            reference["xy"][reference_train],
            reference_queries,
            spatial_transform=spatial_transform,
            hidden_dim=probe["hidden_dim"],
            epochs=probe["epochs"],
            learning_rate=probe["learning_rate"],
            seed=probe["seed"],
            device=DEVICE,
        )
        n_reference_test = len(reference_test)
        heldout_reference_prediction = reference_predictions[:n_reference_test]
        reference_to_update_prediction = reference_predictions[n_reference_test:]

        for protocol, predicted, truth, train_count, train_groups, test_groups in (
            (
                "held_out_reference",
                heldout_reference_prediction,
                reference["xy"][reference_test],
                len(reference_train),
                reference_train_groups,
                reference_test_groups,
            ),
            (
                "reference_to_update",
                reference_to_update_prediction,
                update["xy"][update_test],
                len(reference_train),
                reference_train_groups,
                update_test_groups,
            ),
        ):
            rmse, mean_error = _score(predicted, truth)
            rows.append({
                "run": run_name,
                "stage": stage,
                "representation": representation,
                "protocol": protocol,
                "train_scans": train_count,
                "train_locations": train_groups,
                "test_scans": len(truth),
                "test_locations": test_groups,
                "position_rmse": rmse,
                "mean_position_error": mean_error,
            })

        # Update labels fit only this temporary diagnostic probe; update_test stays held out.
        update_prediction = train_reference_location_mlp_predictions(
            update_features[update_train],
            update["xy"][update_train],
            update_features[update_test],
            spatial_transform=spatial_transform,
            hidden_dim=probe["hidden_dim"],
            epochs=probe["epochs"],
            learning_rate=probe["learning_rate"],
            seed=probe["seed"],
            device=DEVICE,
        )
        rmse, mean_error = _score(update_prediction, update["xy"][update_test])
        rows.append({
            "run": run_name,
            "stage": stage,
            "representation": representation,
            "protocol": "update_to_update",
            "train_scans": len(update_train),
            "train_locations": update_train_groups,
            "test_scans": len(update_test),
            "test_locations": update_test_groups,
            "position_rmse": rmse,
            "mean_position_error": mean_error,
        })

    return rows


## Run the comparisons

The bootstrap AE/GNN row uses the original saved reference-only representations. It fits and scores on different reference coordinate groups. This is a transductive readout because all those reference nodes, including their coordinate inputs to the GNN, were present during bootstrap GNN training.

For online round 1 and post-transition round 2, the reference-trained probe scores both held-out references and held-out updates. A separate temporary probe uses update training labels and scores the exact same held-out update locations.

The update-to-update result is an **upper-bound diagnostic for decodability with some update labels**. It is not the unsupervised GuFu result.


In [3]:
result_rows = []

for run_name, output_root in OUTPUT_ROOTS.items():
    if not output_root.is_dir():
        raise FileNotFoundError(f"Saved output directory not found: {output_root}")
    print(f"Loading saved representations for {run_name}...")

    bootstrap_reference, bootstrap_probe, bootstrap_spatial_transform = (
        _load_bootstrap_reference_data(run_name, output_root)
    )
    result_rows.extend(
        _run_bootstrap_probe_comparisons(
            run_name,
            bootstrap_reference,
            bootstrap_probe,
            bootstrap_spatial_transform,
        )
    )

    bootstrap_xy_by_id = dict(zip(bootstrap_reference["ids"], bootstrap_reference["xy"]))
    for stage in STAGES:
        reference, update, spatial_transform, probe = _load_stage_data(
            run_name, output_root, stage
        )
        if set(reference["ids"]) != set(bootstrap_reference["ids"]):
            raise ValueError(
                f"{run_name}/{stage}: reference IDs differ from the bootstrap baseline."
            )
        for scan_id, xy in zip(reference["ids"], reference["xy"]):
            if not np.allclose(xy, bootstrap_xy_by_id[scan_id], rtol=0.0, atol=1e-8):
                raise ValueError(
                    f"{run_name}/{stage}: reference coordinates differ from the bootstrap baseline."
                )
        result_rows.extend(
            _run_probe_comparisons(
                run_name, stage, reference, update, spatial_transform, probe
            )
        )

results = pd.DataFrame(result_rows)
stage_order = {
    "bootstrap_reference_only": 0,
    "online_round1": 1,
    "post_transition_round2": 2,
}
results["_stage_order"] = results["stage"].map(stage_order)
display(
    results.sort_values(
        ["run", "_stage_order", "representation", "protocol"]
    ).drop(columns="_stage_order").reset_index(drop=True)
)

print("Position RMSE by protocol (lower is better):")
pivot = results.pivot_table(
    index=["run", "stage", "representation"],
    columns="protocol",
    values="position_rmse",
).reset_index()
pivot["_stage_order"] = pivot["stage"].map(stage_order)
display(pivot.sort_values(["run", "_stage_order"]).drop(columns="_stage_order"))


Loading saved representations for GF_from1F_12...
Loading saved representations for GF_fromGF_12...


,run,stage,representation,protocol,train_scans,train_locations,test_scans,test_locations,position_rmse,mean_position_error
0,GF_from1F_12,bootstrap_reference_only,ae,held_out_reference,193,193,65,65,30.848599,23.367060
1,GF_from1F_12,bootstrap_reference_only,gnn,held_out_reference,193,193,65,65,35.855812,32.257918
2,GF_from1F_12,online_round1,ae,held_out_reference,193,193,65,65,30.848599,23.367060
3,GF_from1F_12,online_round1,ae,reference_to_update,193,193,105,105,49.102757,37.110341
4,GF_from1F_12,online_round1,ae,update_to_update,314,314,105,105,29.766564,24.827154
5,GF_from1F_12,online_round1,gnn,held_out_reference,193,193,65,65,40.036738,36.164777
6,GF_from1F_12,online_round1,gnn,reference_to_update,193,193,105,105,45.878201,40.096212
7,GF_from1F_12,online_round1,gnn,update_to_update,314,314,105,105,42.730190,37.439813
8,GF_from1F_12,post_transition_round2,ae,held_out_reference,193,193,65,65,23.010106,19.586647
9,GF_from1F_12,post_transition_round2,ae,reference_to_update,193,193,105,105,81.551005,61.526549


Position RMSE by protocol (lower is better):


protocol,run,stage,representation,held_out_reference,reference_to_update,update_to_update
0,GF_from1F_12,bootstrap_reference_only,ae,30.848599,NaN,NaN
1,GF_from1F_12,bootstrap_reference_only,gnn,35.855812,NaN,NaN
2,GF_from1F_12,online_round1,ae,30.848599,49.102757,29.766564
3,GF_from1F_12,online_round1,gnn,40.036738,45.878201,42.730190
4,GF_from1F_12,post_transition_round2,ae,23.010106,81.551005,32.534033
5,GF_from1F_12,post_transition_round2,gnn,48.515538,58.591955,45.598489
6,GF_fromGF_12,bootstrap_reference_only,ae,27.205842,NaN,NaN
7,GF_fromGF_12,bootstrap_reference_only,gnn,47.837017,NaN,NaN
8,GF_fromGF_12,online_round1,ae,27.205990,41.568065,30.523418
9,GF_fromGF_12,online_round1,gnn,77.088634,158.851076,56.411818


## How to read the pattern

- **Bootstrap AE good, bootstrap GNN poor:** location structure is present in the initial AE latents but is weak or hard to read after the initial GNN.
- **Bootstrap representations are good, later GNN stages worsen:** the loss of readable location signal is associated with streaming adaptation.
- **Held-out reference is poor for both representations:** even within the reference domain, the saved features and this probe setup provide weak localization.
- **Reference→update is poor, but update→update is much better:** update scans contain location information a probe can learn, but the reference-trained mapping does not transfer.
- **GNN update→update is worse than AE update→update:** the GNN stage may be obscuring location information that remains in the AE latents.
- **Both update→update scores are poor:** even a probe trained on some update locations cannot decode much location structure from these vectors.

These comparisons are diagnostic, not causal proof. The held-out-reference scores are transductive because the representations were generated with all reference nodes present; in particular, the GNN receives reference coordinates as input features. The update-trained probe uses labels and a random location-group split, so nearby distinct locations may occur on both sides. All probes use the same MLP architecture and saved updater settings.
